# 03 - Forecasting: Predicting Flares Before They Happen

**Forecasting** answers "will a flare happen in the next 30 minutes?" - the harder, machine-learning part. This notebook is a small **demo** so it runs on a normal computer: it trains on ~40 days of SoLEXS data for a few rounds. The *real* training uses all of GOES then fine-tunes, via the command line:
```
python model/train.py --phase pretrain
python model/train.py --phase finetune
```
**Needs:** processed SoLEXS parquet + `master_catalog.parquet`.

In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import torch, yaml
from pathlib import Path

from model.cnn_lstm import build_model
from model.features import build_feature_series, build_windows, normalize_features, FEATURE_NAMES, FLARE_CLASS_ORDER
from model.evaluate import predict_proba, threshold_sweep, compute_lead_times
from model.train import train
from pipeline.catalog import load_master_catalog

DATA = Path('../data')
with open('../config.yaml') as f:
    cfg = yaml.safe_load(f)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device, '| n_classes:', cfg['forecasting'].get('n_classes', 0))

## 1. Turn light curves into training examples

A machine-learning model learns from **examples**. Here each example is:
- **Input**: a **60-minute window** of the light curve, summarized as **7 numbers per minute** (the "features": brightness, how fast it's rising, the hard/soft ratio, and so on).
- **Label**: 1 if a flare peaks in the **next 30 minutes**, else 0 - plus *which class* it is.

`build_windows` slides along the data creating these windows. `y_cls` is the class label (**-1 means "no flare"**, which training simply ignores). We use only 40 days to keep it small and fast.

In [ ]:
cat = load_master_catalog('../data/processed/master_catalog.parquet')
flare_times = pd.DatetimeIndex(cat['peak_time'])
flare_cls = cat['flare_class'].to_numpy()

fc = cfg['forecasting']
win, lead = fc['input_window_min'], fc['lead_time_min']
sxr_files = sorted(DATA.glob('processed/*_solexs.parquet'))[:40]

Xs, ys, cs = [], [], []
for f in sxr_files:
    sx = pd.read_parquet(f)
    feat = build_feature_series(sx, None)
    m = (flare_times >= feat.index[0]) & (flare_times <= feat.index[-1])
    X, y, yc = build_windows(feat, pd.DatetimeIndex(flare_times[m]), win, lead,
                             flare_classes=list(flare_cls[np.asarray(m)]))
    if len(X):
        Xs.append(X); ys.append(y); cs.append(yc)

assert Xs, 'No windows built - run scripts/process_all.py and check data/processed has SoLEXS parquet.'
X = np.concatenate(Xs); y = np.concatenate(ys); y_cls = np.concatenate(cs)
print(f'X {X.shape}  y {y.shape}  positives {y.mean():.1%}')
print('class counts (positive windows):',
      {FLARE_CLASS_ORDER[i]: int((y_cls == i).sum()) for i in range(len(FLARE_CLASS_ORDER))})

## 2. Do the features actually distinguish flares?

A quick gut-check before training: for each of the 7 features, compare its average value in **flare** windows vs **quiet** windows. If the red and blue bars differ a lot, that feature carries useful signal the model can learn from. If they're identical, the feature is probably useless.

In [ ]:
pos_mean = X[y == 1].mean(axis=(0, 1))
neg_mean = X[y == 0].mean(axis=(0, 1))
fig, ax = plt.subplots(figsize=(10, 4))
xi = np.arange(len(FEATURE_NAMES))
ax.bar(xi - 0.2, pos_mean, 0.4, label='flare windows', color='red', alpha=0.7)
ax.bar(xi + 0.2, neg_mean, 0.4, label='quiet windows', color='blue', alpha=0.7)
ax.set_xticks(xi); ax.set_xticklabels(FEATURE_NAMES, rotation=30, ha='right')
ax.set_title('Mean feature value: flare vs quiet'); ax.legend()
plt.tight_layout(); plt.show()

## 3. Train the model

**Training** = the model repeatedly guesses, checks each guess against the true label, and nudges its internal numbers to be more right. One full pass over the data is an **epoch** (we do 5 here; real training does more). The **loss** number should fall as it learns. Because `config.yaml` has `n_classes: 5`, the model learns *two* things at once: "flare or not" and "which class". We save it to a **separate** file so it won't overwrite your real trained model.

In [ ]:
model = build_model(cfg, device)
train(model, X, y, epochs=5, lr=1e-3, ckpt_path='../models/cnn_lstm_notebook.pt', device=device, y_cls=y_cls)

## 4. How good is it? (the scores)

Plain accuracy is misleading when flares are rare (a model that always says "no flare" looks ~95% accurate but is useless). So we use fairer scores:
- **TPR** (true positive rate): of real flares, what fraction did we catch?
- **FAR** (false alarm rate): of quiet times, what fraction did we wrongly alarm on?
- **TSS = TPR - FAR**: the standard flare-forecasting skill score (0 = no skill, 1 = perfect).
- **HSS**: skill compared to random guessing.
- **AUC**: overall ranking quality (0.5 = coin flip, 1.0 = perfect).

We check these at a few alert thresholds. (This tiny demo won't score well - real training does.)

In [ ]:
n_test = max(1, int(len(X) * 0.2))
X_tr, X_te = X[:-n_test], X[-n_test:]
y_te = y[-n_test:]
X_tr_n, X_te_n, mn, st = normalize_features(X_tr, X_te)
y_prob = predict_proba(model, X_te_n, device=device)
print(threshold_sweep(y_te, y_prob).to_string(index=False))

## 5. ROC curve

The **ROC curve** plots "flares caught" (TPR) against "false alarms" (FAR) as we slide the alert threshold. A curve that bows toward the top-left is good; the diagonal dashed line is random guessing. **AUC** is the area under it - a single summary number.

In [ ]:
from sklearn.metrics import roc_curve, auc
fpr, tpr, _ = roc_curve(y_te, y_prob)
plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, lw=2, label=f'AUC = {auc(fpr, tpr):.3f}')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Alarm Rate'); plt.ylabel('True Positive Rate')
plt.title('ROC — Solar Flare Forecaster'); plt.legend()
plt.tight_layout(); plt.show()

## 6. See the forecast on a real day

Finally we run the model across one full day, minute by minute (the same code the live dashboard uses). The first chart is the **probability of a flare in the next 30 min** over the day, with the alert threshold marked. The second chart - because we trained the multi-class model - shows the probability of **each intensity class** (A/B/C/M/X) at each moment.

In [ ]:
from dashboard.app import run_forecaster
sample = sorted(DATA.glob('processed/*_solexs.parquet'))[0]
feat = build_feature_series(pd.read_parquet(sample), None)
prob_series, class_prob_df = run_forecaster(feat, model, win)

fig, ax = plt.subplots(figsize=(13, 3))
ax.plot(prob_series.index, prob_series.values, color='#ff8844', label='P(flare in next 30 min)')
ax.axhline(cfg['forecasting']['alert_threshold'], color='k', ls='--', label='alert threshold')
ax.set_ylim(0, 1); ax.set_title(f'Forecast — {sample.name[:8]}'); ax.legend()
plt.tight_layout(); plt.show()

if class_prob_df is not None:
    fig, ax = plt.subplots(figsize=(13, 3))
    for c in class_prob_df.columns:
        ax.plot(class_prob_df.index, class_prob_df[c], lw=1, label=c)
    ax.set_ylim(0, 1); ax.set_title('Intensity-class probabilities (A/B/C/M/X)')
    ax.legend(ncol=5); plt.tight_layout(); plt.show()
else:
    print('Binary model (n_classes=0) — no per-class output.')